In [2]:
pip install --no-cache-dir -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

In [4]:
def extract_taxi_parquet(file_path):
    # Parquet files are read in one line
    df = pd.read_parquet(file_path)
    return df

In [12]:
# Define the range of months (1 to 4)
# Using f-string formatting to generate '01', '02', etc.
file_list = [f'dataset/yellow/raw/yellow_tripdata_2026-{month:02d}.parquet' for month in range(1, 5)]

# Extract and combine into one DataFrame
df = pd.concat([extract_taxi_parquet(file) for file in file_list], ignore_index=True)
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.20,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.90,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.70,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.70,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.50,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14908441,2,2026-04-30 23:02:00,2026-04-30 23:08:00,NaN,1.01,NaN,NaN,239,142,0,16.16,0.00,0.5,2.00,0.0,1.0,22.16,NaN,NaN,0.00
14908442,2,2026-04-30 23:08:31,2026-04-30 23:33:16,NaN,5.36,NaN,NaN,211,239,0,42.56,0.00,0.5,0.00,0.0,1.0,47.31,NaN,NaN,0.75
14908443,2,2026-04-30 23:10:19,2026-04-30 23:31:11,NaN,3.07,NaN,NaN,256,61,0,20.60,0.00,0.5,0.00,0.0,1.0,22.10,NaN,NaN,0.00
14908444,2,2026-04-30 23:46:47,2026-05-01 00:07:22,NaN,3.57,NaN,NaN,25,225,0,28.05,0.00,0.5,0.00,0.0,1.0,35.16,NaN,NaN,0.00


In [11]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='str')

In [17]:
def transform_taxi_data(df):
    # 1. Cast Dates
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    
    # 2. Clean Metrics
    # Ensure distance is positive
    df = df[df['trip_distance'] > 0]
    
    # 3. Handle specific Flag columns
    # Convert 'Y'/'N' to 1/0 for SQL friendliness
    df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'Y': 1, 'N': 0})
    
    # 4. Fill NaNs in monetary columns
    cols = ['fare_amount', 'tip_amount', 'total_amount', 'passenger_count', 'Airport_fee', 'cbd_congestion_fee']
    df[cols] = df[cols].fillna(0)
    
    return df

In [18]:
processed = transform_taxi_data(df)
processed

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,0.0,239,238,1,7.20,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,0.0,163,162,2,7.90,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,0.0,43,237,1,10.70,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,0.0,142,209,1,38.70,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,0.0,88,144,1,13.50,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14908441,2,2026-04-30 23:02:00,2026-04-30 23:08:00,0.0,1.01,NaN,NaN,239,142,0,16.16,0.00,0.5,2.00,0.0,1.0,22.16,NaN,0.0,0.00
14908442,2,2026-04-30 23:08:31,2026-04-30 23:33:16,0.0,5.36,NaN,NaN,211,239,0,42.56,0.00,0.5,0.00,0.0,1.0,47.31,NaN,0.0,0.75
14908443,2,2026-04-30 23:10:19,2026-04-30 23:31:11,0.0,3.07,NaN,NaN,256,61,0,20.60,0.00,0.5,0.00,0.0,1.0,22.10,NaN,0.0,0.00
14908444,2,2026-04-30 23:46:47,2026-05-01 00:07:22,0.0,3.57,NaN,NaN,25,225,0,28.05,0.00,0.5,0.00,0.0,1.0,35.16,NaN,0.0,0.00
